# Bài 5: Mô-men ảnh (Image Moments)

Khi đã tách được một đốm (Bài 4), mô-men (moments) cho phép ta tóm tắt hình dạng của nó bằng một vài con số: diện tích, trọng tâm, hướng, và thậm chí cả một mô tả không đổi khi tịnh tiến, co giãn và xoay. Mô-men là một phương án cổ điển, gọn nhẹ thay cho các đặc trưng học được (sẽ đề cập sau) trong bài toán so khớp hình dạng đơn giản.

In [ ]:
# Download the sample images used in this lesson, if not already present
import os, urllib.request

for _f in ['dog_clipart.png', 'cat_clipart.png']:
    _path = f"../img/{_f}"
    if not os.path.exists(_path):
        os.makedirs("../img", exist_ok=True)
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/sbirchfield/sbirchfield.github.io/main/cvintro/img/{_f}", _path)

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

## Mô-men là gì? (What is a moment?)
Với ảnh nhị phân, mô-men thô (raw moment) $m_{pq}$ được định nghĩa là

$$m_{pq} = \sum_{x,y} x^p y^q \, I(x, y)$$

trong đó $I(x,y)$ bằng 1 bên trong hình và bằng 0 ở chỗ khác. Một vài trường hợp đặc biệt chính là những đại lượng quen thuộc:

- $m_{00}$ = diện tích (số điểm ảnh)
- ($\bar{x}, \bar{y}) = (m_{10}/m_{00}, m_{01}/m_{00})$ cho ra trọng tâm

**Bậc (order)** của một mô-men là $p+q$. Mô-men bậc không là $m_{00}$; các mô-men bậc một là $m_{10}$ và $m_{01}$; và các mô-men bậc hai là $m_{20}$, $m_{02}$ và $m_{11}$. `cv2.moments` tính tất cả các giá trị này trong một lần gọi, cùng với các mô-men *trung tâm (central)* $\mu_{ij}$, bất biến với phép tịnh tiến; và các *mô-men Hu (Hu moments)*, bất biến với tịnh tiến, co giãn và xoay.

## Tính mô-men thô bằng tay (Raw moments, computed by hand)
Công thức trên chỉ là một tổng có trọng số theo tọa độ điểm ảnh, nên với ảnh nhị phân nó chẳng qua là: tìm tọa độ $(x,y)$ của mọi điểm ảnh tiền cảnh, rồi cộng $x^py^q$ trên các điểm đó. Trong ảnh động này, lần lượt từng điểm ảnh tiền cảnh được thăm, ngay lập tức đóng góp vào các tổng đang tích lũy, và kết thúc bằng trọng tâm (đánh dấu bằng ngôi sao).

<img src="../animations/moments_demo.gif" alt="Ảnh động minh họa cách tính các mô-men thô m00, m10, m01 và trọng tâm trên một ảnh nhị phân nhỏ" style="display: block; margin: 0 auto; max-width: 550px;">

Tạo một ảnh nhị phân 3x3 rất nhỏ có 3 điểm ảnh tiền cảnh. Tính trực tiếp các mô-men bậc một bằng NumPy và đối chiếu với `cv2.moments`.

In [ ]:
im_tiny = np.array([
    [0, 1, 0],
    [1, 1, 0],
    [0, 0, 0],
], dtype=np.uint8)

ys, xs = np.nonzero(im_tiny)       # (row, col) = (y, x) of every foreground pixel

m00 = len(xs)                 # area: just a count of foreground pixels
m10 = xs.sum()                # sum of x^1 y^0 over foreground pixels
m01 = ys.sum()                # sum of x^0 y^1 over foreground pixels

m = cv2.moments(im_tiny, binaryImage=True)

print(f'foreground pixels (x, y): {[(int(x), int(y)) for x, y in zip(xs, ys)]}')
print(f'moments:     m00 = {m00}   m10 = {m10}   m01 = {m01}')
print(f"cv2.moments: m00 = {m['m00']:.0f}   m10 = {m['m10']:.0f}   m01 = {m['m01']:.0f}  <-- same as previous line")
print(f'centroid = ({m10 / m00:.2f}, {m01 / m00:.2f})')

## Một đốm thuôn dài, bị xoay (An elongated, rotated blob)
Hãy vẽ một hình elip bị xoay để có thể dễ dàng nhìn ra hướng của nó và đối chiếu với kết quả tính từ mô-men.

In [ ]:
im_binary1 = np.zeros((200, 200), dtype=np.uint8)
pixval = 1  # or 255, but note that it will make the moments bigger 
center = (100, 100)
axes_len = (70, 25)
angle_deg = 30
cv2.ellipse(im_binary1, center, axes_len, angle_deg, 0, 360, pixval, cv2.FILLED)

plt.imshow(255 * im_binary1, cmap='gray')
plt.title(f'Ellipse drawn at {angle_deg} degrees')
plt.axis('off')
plt.show()

Hàm `cv2.ellipse` của OpenCV tạo ra những hiệu ứng lạ quanh đường biên. Nếu muốn khắc phục, bạn có thể khá dễ dàng tự viết code vẽ elip.

In [ ]:
def draw_ellipse(im, cen, axes, angle_deg, val):
    h, w = im.shape
    Y, X = np.mgrid[0:h, 0:w]
    ang = np.deg2rad(angle_deg)
    XX = np.cos(ang) * (X-cen[0]) + np.sin(ang) * (Y-cen[1])
    YY = -np.sin(ang) * (X-cen[0]) + np.cos(ang) * (Y-cen[1])
    return np.uint8(val * (np.sqrt( (XX**2) + ((axes[0] / axes[1])**2)*(YY**2)) < (1.006)*axes[0]))

im_binary = np.zeros((200, 200), dtype=np.uint8)
im_binary = draw_ellipse(im_binary, center, axes_len, angle_deg, pixval)

cv_mask = im_binary1 > 0
fixed_mask = im_binary > 0
cv_only = (cv_mask & ~fixed_mask).astype(np.uint8)
fixed_only = ~cv_mask & fixed_mask

diff = np.full((200, 200, 3), 255, dtype=np.uint8)
diff[cv_only > 0] = [255, 60, 60]   # cv2.ellipse only -> bright red
diff[fixed_only] = [60, 160, 255]   # fixed ellipse only -> bright blue (rare/none here)

fig, axes = plt.subplots(1, 3, figsize=(10, 4))
axes[0].imshow(255 * im_binary, cmap='gray')
axes[0].set_title('Fixed ellipse', fontsize=9)
axes[0].axis('off')
axes[1].imshow(255 * im_binary1, cmap='gray')
axes[1].set_title('cv2.ellipse', fontsize=9)
axes[1].axis('off')
axes[2].imshow(diff)
axes[2].set_title('Difference (red = cv2.ellipse only)', fontsize=9)
axes[2].axis('off')
plt.tight_layout()
plt.show()

## Tính mô-men thô (Computing raw moments)
Tính mô-men thô rất dễ: chỉ cần lấy tổng trên các điểm ảnh của ảnh (như đã giải thích ở trên).

In [ ]:
def compute_moments(im):
    summ = 0
    sumx = 0
    sumy = 0    
    for y in range(im.shape[0]):
        for x in range(im.shape[1]):
            v = im[y,x]
            if v > 0:
                summ += 1
                sumx += x
                sumy += y
    out = {}
    out['m00'] = summ
    out['m10'] = sumx
    out['m01'] = sumy
    return out

mm = compute_moments(im_binary)
m = cv2.moments(im_binary, binaryImage=True)
print(f"moments:     m00 = {mm['m00']:.0f}  m10 = {mm['m10']:.0f}  m01 = {mm['m01']:.0f}")
print(f"cv2.moments: m00 = {m['m00']:.0f}  m10 = {m['m10']:.0f}  m01 = {m['m01']:.0f}  <-- same as previous line")

## Diện tích và trọng tâm từ mô-men (Area and centroid from moments)

In [ ]:
m = cv2.moments(im_binary, binaryImage=True)

area = m['m00']
cx = m['m10'] / m['m00']
cy = m['m01'] / m['m00']

print(f'area (m00)   = {area:.0f} pixels')
print(f'centroid     = ({cx:.1f}, {cy:.1f})')

plt.imshow(im_binary, cmap='gray')
plt.scatter(cx, cy, c='red', marker='x', s=80)
plt.title('Centroid from moments')
plt.axis('off')
plt.show()

## Mô-men trung tâm (Central moments)
Mô-men thô $m_{pq}$ phụ thuộc vào vị trí ngẫu nhiên mà hình nằm trong ảnh &mdash; di chuyển hình thì mọi $m_{ij}$ (trừ $m_{00}$) đều thay đổi. **Mô-men trung tâm (central moments)** khắc phục điều này bằng cách đo quanh chính trọng tâm $(\bar x, \bar y)$ của hình thay vì gốc tọa độ của ảnh:

$$\mu_{pq} = \sum_{x,y} (x-\bar x)^p\,(y-\bar y)^q\, I(x,y)$$

Tính lại tổng này từ đầu nghĩa là phải duyệt lại mọi điểm ảnh một lần nữa. Thay vào đó, có một đẳng thức dời gốc &mdash; phiên bản mô-men ảnh của $\mathrm{Var}(X) = E[X^2] - E[X]^2$ &mdash; cho phép tính mô-men trung tâm trực tiếp từ mô-men thô:

$$\mu_{20} = m_{20} - \bar x\,m_{10}, \qquad \mu_{02} = m_{02} - \bar y\,m_{01}, \qquad \mu_{11} = m_{11} - \bar x\,m_{01} = m_{11} - \bar y\,m_{10}$$

`cv2.moments` trả về cả mô-men thô (`m['m20']`, ...) lẫn mô-men trung tâm (`m['mu20']`, ...) trong cùng một dictionary, nên trên thực tế bạn không bao giờ cần tính tay các giá trị này.

In [ ]:
mu20_manual = m['m20'] - cx * m['m10']
mu02_manual = m['m02'] - cy * m['m01']
mu11_manual_v1 = m['m11'] - cx * m['m01']
mu11_manual_v2 = m['m11'] - cy * m['m10']

print(f'moments:     mu20 = {mu20_manual:.1f}   mu02 = {mu02_manual:.1f}   mu11 = {mu11_manual_v1:.1f} = {mu11_manual_v2:.1f}')
print(f"cv2.moments: mu20 = {m['mu20']:.1f}   mu02 = {m['mu02']:.1f}   mu11 = {m['mu11']:.1f}")

## Hướng từ mô-men trung tâm (Orientation from central moments)
Các mô-men trung tâm $\mu_{20}$, $\mu_{02}$, $\mu_{11}$ mô tả độ trải của hình quanh trọng tâm — về bản chất chính là ma trận hiệp phương sai (covariance matrix) của nó (sẽ được bàn lại ở Bài 6). Góc của trục chính (hướng có độ trải lớn nhất) liên hệ với các mô-men như sau:

$$\tan 2\theta = \left(\frac{2\mu_{11}}{\mu_{20} - \mu_{02}}\right)$$

Khi giải ngược phương trình này, hãy cẩn thận dùng `atan2` (hàm arctang nhận hai tham số riêng biệt) để đảm bảo góc đầu ra nằm đúng góc phần tư:

$$\theta = \frac{1}{2}\,\mathrm{atan2}\!\left(2\mu_{11},\; \mu_{20} - \mu_{02}\right)$$

In [ ]:
theta = 0.5 * np.arctan2(2 * m['mu11'], m['mu20'] - m['mu02'])
theta_deg = np.degrees(theta)

print(f'orientation from moments = {theta_deg:.1f} degrees')
print(f'angle used to draw the ellipse = {angle_deg} degrees')

# The eigenvalues of the (normalized) covariance matrix of central moments
# give the axis lengths of the equivalent ellipse: axis = 4*sqrt(eigenvalue).
cov = np.array([[m['mu20'], m['mu11']], [m['mu11'], m['mu02']]]) / m['m00']
eigvals, _ = np.linalg.eigh(cov)
semi_major = 2 * np.sqrt(eigvals[-1])

theta = np.deg2rad(angle_deg)  ####################
dx, dy = semi_major * np.cos(theta), semi_major * np.sin(theta)

print(semi_major, cx, cy, dx, dy)  ######################
plt.imshow(im_binary, cmap='gray')
plt.plot([cx - dx, cx + dx], [cy - dy, cy + dy], c='red', linewidth=2)
plt.scatter(cx, cy, c='red', marker='x', s=80)
plt.title('Major axis recovered from moments')
plt.axis('off')
plt.axis('equal')
plt.show()

## Mô-men Hu: bộ mô tả hình dạng bất biến với tư thế (Hu moments: a shape descriptor invariant to pose)
**Mô-men Hu** (`cv2.HuMoments`) là 7 giá trị tính từ các mô-men trung tâm, (gần như) không đổi bất kể vị trí, kích thước và góc xoay của hình. Điều này khiến chúng hữu ích để so sánh hai hình mà không cần căn chỉnh trước.

Để thấy điều đó, ta tạo ba phiên bản của một hình con chó: ở tư thế ban đầu, bị tịnh tiến, và bị xoay + co giãn. Ta so sánh mô-men Hu của các phiên bản này với nhau, và với một hình khác (con mèo).

In [ ]:
def warp(im, mat):
    rows, cols = im.shape[:2]
    return cv2.warpAffine(im, np.float32(mat), (cols, rows))

im_original = cv2.imread('../img/dog_clipart.png', cv2.IMREAD_GRAYSCALE)
im_translated = warp(im_original, [[1, 0, 10], [0, 1, 20]])
im_rotated_scaled = warp(im_original, cv2.getRotationMatrix2D([60, 60], angle=45, scale=0.5))
im_different_shape = cv2.imread('../img/cat_clipart.png', cv2.IMREAD_GRAYSCALE)

fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for ax, img, title in zip(
    axes,
    [im_original, im_translated, im_rotated_scaled, im_different_shape],
    ['Original', 'Translated', 'Rotated + scaled', 'Different shape'],
):
    ax.imshow(img, cmap='gray')
    ax.set_title(title, fontsize=10)
    ax.axis('off')
plt.tight_layout()
plt.show()

<p class="image-attribution">Nguồn ảnh: <a href="https://www.publicdomainpictures.net" target="_blank" rel="noopener">Public domain pictures</a></p>

Bây giờ hãy tính các mô-men Hu.

In [ ]:
def hu_log(im):
    m = cv2.moments(im, binaryImage=True)
    hu = cv2.HuMoments(m).flatten()
    # log-scale since raw Hu moments span many orders of magnitude
    return -np.sign(hu) * np.log10(np.abs(hu) + 1e-30)

# Format with a fixed width (":8.3f") to align the values across rows.
label_width, col_width, n_hu = 18, 8, 7
print(' ' * (label_width + 2) + 'Hu moments'.center(col_width * n_hu))
print(f'{"image":>{label_width}}  ' + ''.join(f'{i:>{col_width}}' for i in range(1, n_hu + 1)))
for im, name in [
    (im_original, 'original'),
    (im_translated, 'translated'),
    (im_rotated_scaled, 'rotated+scaled'),
    (im_different_shape, 'different shape'),
]:
    row = ''.join(f'{v:{col_width}.3f}' for v in hu_log(im))
    print(f'{name:>{label_width}}  {row}')

Lưu ý rằng các phiên bản bị tịnh tiến và bị xoay+co giãn cho ra mô-men Hu tương tự nhau, trong khi hình khác cho ra giá trị khác hẳn. Đây chính là tính bất biến khiến mô-men Hu hữu ích cho việc so khớp hình dạng.

### Bài tập (Exercises)
1. Dùng `cv2.findContours` để lấy đường viền của một đốm trong ảnh nhị phân ở Bài 4, rồi gọi `cv2.moments` trên *đường viền (contour)* thay vì trên toàn bộ mặt nạ nhị phân. So sánh trọng tâm với trọng tâm tính ở đây.
2. Vẽ một hình bị lật gương thay vì bị xoay. Mô-men Hu của nó có còn gần với hình gốc không? (Gợi ý: hãy nghĩ xem mô-men nắm bắt hoặc không nắm bắt thông tin định thức/tính chẵn lẻ (parity) nào.)